# 计算每个cluster中真实细胞类型占比

In [ ]:
import pandas as pd
from scanpy import read_h5ad

# 如果数据是 h5ad 文件
adata = read_h5ad("ac9c13da_pca_0.20.h5ad")
df = adata.obs[["leiden_pca_0.20", "cell_type"]].copy()

# 如果数据是 CSV 文件
# df = pd.read_csv("your_data.csv")

# (1) 计算每个聚类的细胞数量
cluster_counts = df["leiden_pca_0.20"].value_counts().sort_index()

# (2) 计算每个聚类中细胞类型的占比
celltype_stats = (
    df.groupby("leiden_pca_0.20")["cell_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .unstack()
    .fillna(0)
)

# (3) 按占比从高到低排序（每个聚类内部）
sorted_stats = celltype_stats.apply(
    lambda x: x.sort_values(ascending=False).to_dict(), axis=1
)

# 合并结果
result = pd.DataFrame({
    "Cluster": cluster_counts.index,
    "Total_Cells": cluster_counts.values,
    "CellType_Composition": sorted_stats
})

# 输出到 CSV
result.to_csv("ac9c13da_celltype_stats_pca_0.20.csv", index=False)

# 使用有全部基因的原数据集计算cluster之间的差异基因，根据具体情况添加normalize和log1p的代码

In [ ]:
import pandas as pd
from scanpy import read_h5ad
import scanpy as sc

# 如果数据是 h5ad 文件
adata = read_h5ad("ac9c13da_pca_0.20.h5ad")
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)
sc.tl.rank_genes_groups(adata, groupby="leiden_pca_0.20", method="wilcoxon", use_raw=False)
diff_genes_df = sc.get.rank_genes_groups_df(adata, group=None)
result_df = pd.DataFrame({
    'cluster': diff_genes_df['group'].unique(),
    'top_20_diff_genes': [
        ', '.join(diff_genes_df[diff_genes_df['group'] == cluster]
                 .sort_values('scores', ascending=False)
                 .head(20)['names'].tolist())
        for cluster in diff_genes_df['group'].unique()
    ]
})

# 4. 保存为CSV（保留引号）
result_df.to_csv('ac9c13da_cluster_diff_genes_pca_0.20.csv', index=False, quoting=1)

# cluster中计算每个celltype的Marker基因的表达量之和×overlap / N，选取排名前K的celltype作为推荐注释的celltype

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from itertools import chain
import matplotlib.pyplot as plt
import seaborn as sns
import os

data_path = "ac9c13da_pro.h5ad"
adata = sc.read_h5ad(data_path)

marker_df = pd.read_csv("blood.csv")
cluster_celltype =  pd.read_csv("ac9c13da_cluster_diff_genes_cell.csv")
diff_genes_file = "ac9c13da_cluster_diff_genes_cell.csv"

def calculate_expression_weight(adata, marker_genes):
    # 筛选出存在于adata中的marker基因
    valid_genes = [gene for gene in marker_genes if gene in adata.var_names]
    
    if not valid_genes:
        return 0.0  # 无有效基因时返回0权重
    
    # 计算这些基因的平均表达量（稀疏矩阵安全处理）
    expr_matrix = adata[:, valid_genes].X.toarray()      
    mean_expression = np.mean(expr_matrix, axis=0)  # 按基因求均值
    return np.sum(mean_expression)  # 返回所有基因的平均值
    

def manual_cluster_annotation(adata, marker_df, diff_genes_file, cluster_key="leiden_pca_0.20"):
    # 读取差异基因文件
    diff_genes_df = pd.read_csv(diff_genes_file)
    
    # 将top_20_diff_genes列从字符串转换为列表
    diff_genes_df['top_20_diff_genes'] = diff_genes_df['top_20_diff_genes'].str.split(', ')
    
    cluster_to_celltype = {}
    cluster_to_markers = {}
    
    output_file = "ac9c13da_pca_0.20.txt"
    with open(output_file, 'w') as f:
        # 遍历所有cluster
        for cluster in diff_genes_df['cluster'].unique():
            cluster_str = str(cluster)
            key = adata.obs[cluster_key] == cluster_str
            cluster_adata = adata[key].copy()
            
            # 获取该cluster的差异基因
            cluster_diff_genes = diff_genes_df[diff_genes_df['cluster'] == cluster]['top_20_diff_genes'].iloc[0]
            print(f"\nCluster {cluster} differential genes:", cluster_diff_genes, file=f)
            CellType_Composition = cluster_celltype[cluster_celltype['Cluster'] == cluster]['CellType_Composition'].iloc[0]
            print(f"Cluster {cluster} cellType composition: {CellType_Composition}", file=f)
            
            # 计算与所有细胞类型的匹配分数
            celltype_scores = []
            expr_scores = []
            for idx, row in marker_df.iterrows():
                celltype = row['Low-hierarchy cell types']
                if pd.notna(row['markergene']) and row['markergene']:
                    genes = [g.strip() for g in str(row['markergene']).split(',') if g.strip()]
                    marker_genes = set(genes)
                else:
                    comp_genes = str(row['computational_gene']).split(',')
                    clean_genes = [g.strip() for g in comp_genes if g.strip()]
                    marker_genes = set(clean_genes[:20])
                score = calculate_expression_weight(cluster_adata, marker_genes)
                expr_scores.append((celltype, score, marker_genes))
            
            expr_scores.sort(key=lambda x: x[1], reverse=True)
            top5_candidates = expr_scores

            all_overlaps_zero = True
            for celltype, expr_score, marker_genes in top5_candidates:
                overlap = len(set(cluster_diff_genes) & marker_genes)
                score = overlap / len(marker_genes) * expr_score
                celltype_scores.append((celltype, score, expr_score, overlap, marker_genes))    
                if overlap > 0:
                    all_overlaps_zero = False
            
            # 按分数排序，获取前n个候选
            # celltype_scores.sort(key=lambda x: x[1], reverse=True)
            # top_candidates = celltype_scores[:5]
            # 按分数排序，获取前n个候选
            #celltype_scores.sort(key=lambda x: x[1], reverse=True)
            if all_overlaps_zero:
                # 所有overlap为0时按expr_score降序
                celltype_scores.sort(key=lambda x: x[2], reverse=True)
            else:
                # 否则按score降序，然后expr_score降序
                celltype_scores.sort(key=lambda x: x[1], reverse=True)
            top_candidates = celltype_scores[:10]
            
            # 打印候选信息
            print(f"Top candidate cell types for Cluster {cluster}:", file=f)
            for i, (celltype, score, expr_score, overlap, marker_genes) in enumerate(top_candidates, 1):
                #marker = marker_df[marker_df['Low-hierarchy cell types'] == celltype]['markergene'].tolist()
                #marker = parse_genes(marker)
                print(f"{i}. {celltype} (score: {score:.2f}, expr_score: {expr_score}, overlap: {overlap}, marker: {len(marker_genes)})", file=f)
                print(f"Marker genes for {celltype}: {marker_genes}", file=f)
            
            # 获取最佳匹配的marker genes
            best_match = top_candidates[0][0] if top_candidates else 'Unknown'
            # suggested_markers = marker_df[marker_df['cell_type_ontology_term_id'] == best_match]['markergene'].tolist()
            # suggested_markers = suggested_markers[0] if suggested_markers else []
            # suggested_markers_str = ", ".join(suggested_markers) if suggested_markers else "N/A"
            
            # 用户选择
            print(f"Suggested cell type for Cluster {cluster}: {best_match}", file=f)
            #print(f"Marker genes for {best_match}: {suggested_markers_str}", file=f)
        

manual_cluster_annotation(adata, marker_df, diff_genes_file)

# 根据txt文件生成每个cluster排名第一的注释细胞类型

In [ ]:
# 暴力逐行搜索（适用于格式严格一致的文件）
with open('ac9c13da_pca_0.20.txt', 'r') as f:
    lines = f.readlines()

results = []
current_cluster = None
for line in lines:
    if line.startswith('Cluster '):
        current_cluster = line.split()[1].strip(':')
    if 'Suggested cell type for Cluster' in line:
        celltype = line.split(':')[-1].strip()
        results.append({'Cluster': f'{current_cluster}', 'CellType': celltype})

pd.DataFrame(results).to_csv('ac9c13da_pca_0.20.csv', index=False)

# 将注释的细胞类型映射到h5ad文件中obs的marker_annotation列中，与真实细胞类型（cell_type_ontology_term_id）计算NMI和ARI

In [ ]:
import scanpy as sc
import pandas as pd

# 1. 加载h5ad文件和两个CSV文件
adata = sc.read_h5ad("ac9c13da_pca_0.20.h5ad")  # 替换为h5ad文件路径
csv1 = pd.read_csv("ac9c13da_pca_0.20.csv")  # Cluster到CellType的映射
csv2 = pd.read_csv("blood.csv")  # CellType到ID的映射

# 2. 检查h5ad中是否存在leiden_res_0.20列
if 'leiden_pca_0.20' not in adata.obs.columns:
    raise KeyError("adata.obs中缺少'leiden_res_0.50'列")

# 3. 合并两个CSV文件创建完整映射表
mapping = pd.merge(
    csv1.rename(columns={'Cluster': 'cluster_num'}),
    csv2,
    left_on='CellType',
    right_on='Low-hierarchy cell types',
    how='left'
)

# 4. 将Cluster列转换为字符串类型（与h5ad一致）
mapping['cluster_num'] = mapping['cluster_num'].astype(str)

# 5. 创建cluster到ontology_id的字典
cluster_to_ontology = dict(zip(
    mapping['cluster_num'],
    mapping['cell_type_ontology_term_id']
))

# 6. 添加marker_annotation列
adata.obs['marker_annotation'] = adata.obs['leiden_pca_0.20'].map(cluster_to_ontology)

# 7. 检查未映射的cluster
unmapped_clusters = set(adata.obs['leiden_pca_0.20']) - set(cluster_to_ontology.keys())
if unmapped_clusters:
    print(f"警告：以下cluster未在CSV1中找到映射: {unmapped_clusters}")
    print("将为这些cluster填充NaN")

# 检查必要的列是否存在
required_columns = ['cell_type_ontology_term_id', 'marker_annotation']
for col in required_columns:
    if col not in adata.obs.columns:
        raise KeyError(f"缺少必需的列: {col}")

# 提取真实标签和预测标签
true_labels = adata.obs['cell_type_ontology_term_id'].astype(str)
pred_labels = adata.obs['marker_annotation'].astype(str)

# 过滤无效值（NaN或空字符串）
valid_mask = (~true_labels.isnull()) & (~pred_labels.isnull()) & (true_labels != '') & (pred_labels != '')
true_labels = true_labels[valid_mask]
pred_labels = pred_labels[valid_mask]

if len(true_labels) == 0:
    raise ValueError("无有效标签可用于计算")

# 计算NMI和ARI
nmi = normalized_mutual_info_score(true_labels, pred_labels)
ari = adjusted_rand_score(true_labels, pred_labels)

# 输出结果
print(f"有效细胞数: {len(true_labels)}/{adata.n_obs}")
print(f"NMI (标准化互信息): {nmi:.4f}")
print(f"ARI (调整兰德指数): {ari:.4f}")